# **Homework 05 - Machine Learning - Zoomcamp 2026**

In this homework we don't train a model. We use the frozen lead-scoring release of the 2026 cohort, copied from `cohorts/2026/homework/05-deployment` into this folder: the trained pipeline (`pipeline.bin`), the locked environment (`pyproject.toml`, `uv.lock`, `.python-version`), the API (`predict.py`) and the container definition (`Dockerfile`).

This notebook runs on the repository kernel (Python 3.12, Scikit-Learn 1.9.1), which is **not** the environment the model was trained in (Python 3.11.15, Scikit-Learn 1.7.2). So everything that touches the model runs through `uv run` inside this folder, which uses the release's own locked environment in `.venv`. If the kernel has `VIRTUAL_ENV` set to the repository environment, `uv` prints a warning that it is ignored, which is expected.

## **Imports**

In [1]:
import hashlib
import json
import subprocess
import sys
import time
import tomllib

from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

## **Setup**

Install the locked environment of the release. With `--locked`, `uv` fails if `uv.lock` is out of date with `pyproject.toml` instead of regenerating it, so we get exactly the reference environment:

In [2]:
!uv sync --locked

Resolved 19 packages in 0.63ms
Checked 18 packages in 0.25ms


The smoke test loads the model directly (without the API) and checks that the inference is deterministic:

In [3]:
!uv run python smoke_test.py

{'conversion_probability': 0.769799, 'conversion': True}


Check that the model artifact is the expected one, comparing its SHA-256 checksum with the one given in the homework:

In [4]:
EXPECTED_SHA256 = "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1"

model_sha256 = hashlib.sha256(Path("pipeline.bin").read_bytes()).hexdigest()

print(f"pipeline.bin: {model_sha256}")
print(f"Matches the expected checksum: {model_sha256 == EXPECTED_SHA256}")

pipeline.bin: 1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1
Matches the expected checksum: True


## **Question 01**

- **Problem statement: Run `uv --version` and record the version.**

### **Solution**

In [5]:
!uv --version

uv 0.12.7 (x86_64-unknown-linux-gnu)


This is a setup check, not a graded answer: the local `uv` version is not a property of the model. Note that the container uses its own pinned `uv` (0.10.11, see Question 05).

> **Answer: `uv 0.12.7`.**

## **Question 02**

- **Problem statement: Which Scikit-Learn version is part of the 2026 reference environment: `1.6.1`, `1.7.2`, `1.8.0` or `2.0.0`?**

### **Solution**

`pyproject.toml` declares the direct dependencies of the project, here pinned with `==`:

In [6]:
pyproject = tomllib.loads(Path("pyproject.toml").read_text())

pyproject["project"]["dependencies"]

['fastapi==0.119.1', 'numpy==2.3.3', 'scikit-learn==1.7.2', 'uvicorn==0.38.0']

`uv.lock` records the whole resolved dependency tree, including the transitive dependencies (such as `scipy` and `joblib`, required by Scikit-Learn), each one with its exact version:

In [7]:
lock = tomllib.loads(Path("uv.lock").read_text())

locked_versions = {package["name"]: package["version"] for package in lock["package"]}

for name, version in locked_versions.items():
    print(f"{name:<25} {version}")

annotated-types           0.8.0
anyio                     4.15.1
click                     8.5.0
cloudpickle               3.1.2
fastapi                   0.119.1
h11                       0.16.0
idna                      3.19
joblib                    1.6.0
mlzoomcamp-2026-deployment 0.1.0
numpy                     2.3.3
pydantic                  2.13.5
pydantic-core             2.46.5
scikit-learn              1.7.2
scipy                     1.17.1
starlette                 0.48.0
threadpoolctl             3.6.0
typing-extensions         4.16.0
typing-inspection         0.4.4
uvicorn                   0.38.0


And the version actually installed in the release environment:

In [8]:
!uv run python -c "import sklearn; print(sklearn.__version__)"

1.7.2


> **Answer: Scikit-Learn `1.7.2`.**

## **Question 03**

- **Problem statement: Write a Python script that loads `pipeline.bin` with `pickle` and calls `predict_proba` for the lead below. What's the probability of conversion (rounded to 3 digits)?**

```json
{
  "lead_source": "paid_ads",
  "industry": "technology",
  "employment_status": "employed",
  "location": "north_america",
  "number_of_courses_viewed": 2,
  "annual_income": 79276.0,
  "interaction_count": 4,
  "lead_score": 0.41
}
```

### **Solution**

The pickle doesn't store the code of the model, only its data (the vocabulary of the `DictVectorizer`, the coefficients of the logistic regression) and references to the Scikit-Learn classes. When loading, these classes are imported from the installed Scikit-Learn, so the model must be loaded in the same version it was saved with. Loading it in the notebook kernel (1.9.1) raises an `InconsistentVersionWarning`.

So we write the script to a file and run it with `uv run`, in the release environment. Since the pipeline starts with the `DictVectorizer`, we pass the lead directly as a dictionary:

In [9]:
%%writefile q3_predict.py
import pickle

with open("pipeline.bin", "rb") as f_in:
    pipeline = pickle.load(f_in)

lead = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}

probability = pipeline.predict_proba([lead])[0, 1]
print(f"Probability of conversion: {probability:.3f}")

Overwriting q3_predict.py


In [10]:
!uv run python q3_predict.py

Probability of conversion: 0.533


> **Answer: The probability of conversion is 0.533.**

## **Question 04**

- **Problem statement: Serve the model with the provided API and send the lead below to `POST /predict`. What's the `conversion_probability` (rounded to 3 digits)?**

```json
{
  "lead_source": "organic_search",
  "industry": "technology",
  "employment_status": "employed",
  "location": "europe",
  "number_of_courses_viewed": 4,
  "annual_income": 80304.0,
  "interaction_count": 7,
  "lead_score": 0.74
}
```

### **Solution**

In a terminal we would run `uv run uvicorn predict:app --host 0.0.0.0 --port 9696` and send the request from a second terminal. A `!` command would block the notebook while the server runs, so we start it in the background with `subprocess.Popen` instead.

The server takes a few seconds to start, so we poll `GET /health` until it answers:

In [11]:
BASE_URL = "http://localhost:9696"


def wait_for_health(base_url: str, timeout: float = 60.0) -> dict:
    """Poll GET /health until the API answers or the timeout expires.

    Args:
        base_url: Base URL of the API.
        timeout: Maximum waiting time, in seconds.

    Returns:
        The JSON response of the health endpoint.
    """
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urlopen(f"{base_url}/health") as response:
                return json.load(response)
        except (URLError, ConnectionError):
            time.sleep(1)

    raise TimeoutError(f"The API at {base_url} did not answer within {timeout} s")

In [12]:
# Discard the server logs, so that they don't mix with the outputs of the next cells
server = subprocess.Popen(
    ["uv", "run", "uvicorn", "predict:app", "--host", "0.0.0.0", "--port", "9696"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

wait_for_health(BASE_URL)

{'status': 'ok',
 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1',
 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}

Now we send the lead with `curl`, as in the homework:

In [13]:
%%bash
curl -s http://localhost:9696/predict \
  -H 'Content-Type: application/json' \
  -d '{
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74
  }'

{"conversion_probability":0.769799,"conversion":true}

The response has the probability in `[0, 1]` and the boolean `conversion` field (`True` when the probability is `>= 0.5`). We stop the server to free the port 9696 for the container in Question 06:

In [14]:
server.terminate()
server.wait(timeout=10);

> **Answer: The `conversion_probability` is 0.770.**

## **Question 05**

- **Problem statement: Which Python base-image tag is declared in the Dockerfile: `python:3.9-slim-bullseye`, `python:3.11.15-slim-bookworm`, `python:3.12-slim-bookworm` or `python:3.13.10-slim-bookworm`?**

### **Solution**

In [15]:
print(Path("Dockerfile").read_text())

FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3

COPY --from=ghcr.io/astral-sh/uv:0.10.11@sha256:3472e43b4e738cf911c99d41bb34331280efad54c73b1def654a6227bb59b2b4 /uv /uvx /bin/

ENV PATH="/code/.venv/bin:$PATH" \
    UV_COMPILE_BYTECODE=1 \
    UV_LINK_MODE=copy

WORKDIR /code
COPY pyproject.toml uv.lock .python-version ./
RUN uv sync --locked --no-dev

COPY model.py predict.py feature_defaults.json model_metadata.json pipeline.bin q6_test.py ./

EXPOSE 9696
ENTRYPOINT ["uvicorn", "predict:app", "--host", "0.0.0.0", "--port", "9696"]



The base image is declared in the `FROM` instruction. Besides the tag, it is pinned by its digest (`@sha256:...`): a tag can be republished pointing to another image, while the digest identifies the exact content. The same is done for the `uv` binary, copied from the `ghcr.io/astral-sh/uv:0.10.11` image.

> **Answer: `python:3.11.15-slim-bookworm`.**

## **Question 06**

- **Problem statement: Run the same request of Question 04 against the container. What's the `conversion_probability` (rounded to 3 digits)?**

### **Solution**

Build the image (`-q` only prints the image ID; remove it to see each step of the build):

In [16]:
!docker build -q -t zoomcamp-model:2026-hw5 .

sha256:c6b4cc9f1ce368be2caebcef6ac87f8e0f2a1530a13556b4cdcd519b4704b472


Run the container in the background (`-d`), mapping the port 9696 of the container to the port 9696 of the host. With `--rm` the container is deleted when it stops:

In [17]:
!docker run -d --rm -p 9696:9696 --name hw5-lead-scoring zoomcamp-model:2026-hw5

25d37060a5f05e19773d166cc16d06e202ad6cadb8e4f14be529b406c4ce3fa5


The health endpoint reports the checksum of the model being served, which must match the one recorded in `model_metadata.json`:

In [18]:
health = wait_for_health(BASE_URL)
metadata = json.loads(Path("model_metadata.json").read_text())

print(health)
print(f"Model checksum matches the metadata: {health['model_sha256'] == metadata['model_sha256']}")

{'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}
Model checksum matches the metadata: True


Send the lead with the provided client. `q6_test.py` only uses the standard library, so it doesn't need the release environment and runs with the kernel's Python:

In [19]:
!{sys.executable} q6_test.py

{'conversion_probability': 0.769799, 'conversion': True}


In [20]:
!docker stop hw5-lead-scoring

hw5-lead-scoring


The result is the same as in Question 04: the container runs the same artifact with the same locked dependencies, just in a different place.

> **Answer: The `conversion_probability` is 0.770.**